<a href="https://colab.research.google.com/github/udituen/RAG-LLM/blob/main/agentic_workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Building a simple ReAct agent from scratch with only an LLM and python

In [11]:
from openai import OpenAI
import os
from google.colab import userdata

In [12]:
api_key = userdata.get("OPENAI_API_KEY")

In [13]:
client = OpenAI(api_key=api_key)

In [20]:
response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[{"role":"user","content":"hello"}]
    )
response.choices[0].message.content

'Hello! How can I assist you today?'

In [57]:
# define the agent class

class Agent():
  def __init__(self, system=""):
    self.system = system # store the system prompt
    self.messages = []  # store all conversation history in the messages list. This enable the model keep track of all conversations
    # next, define and store the system prompt in the list
    self.messages.append({"role":"system", "content": system})

  def __call__(self, message) -> str:
    self.messages.append({"role":"user", "content":message})
    result = self.execute()
    self.messages.append({"role":"assistant","content":result})
    return result

  def execute(self):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=self.messages
    )
    return response.choices[0].message.content

  def reset(self):
    self.messages= self.messages[{"role":"system", "content":self.system}] if self.system else []





In [97]:
prompt = """
You are a personal assistant. Your task is to book a flight depending on the weather conditions.
You should follow these reasoning steps: Thought, Action, PAUSE, Response.

Thought: reinterate what the user wants, specifying source location, destination location and time.
Action: call the weather function to ascertain the current weather of the destination location.
        if the weather is higher than 5 degrees celcius, then call the flight schedule function to book the flight, else tell the user the weather condition is not suitable for travel.
Observation will be the result of running those actions.

Your available actions are:
weather: destination, time. e.g ottawa, 2pm
returns the temperature of destination at time
book: source, destination, time
returns the flight details

Example:
user: Book a flight going from Toronto to Ottawa by 2pm today.
Thought: The user intends to travel from Toronto which is the source location, to Ottawa which is the destination location, by 2pm time today.
         I need to find out what the temperate would be at 2pm .
Action: weather: Ottawa, 2pm
PAUSE

You will be called again with this:
Result: The temperature is 5 degrees

ONLY If Result is less than 5 degrees, stop executing with
Answer: the weather condition is not suitable for travel as it is less than 5 degrees celsius.

Else, run the next step
Action: book: Toronto, Ottawa, 2pm
PAUSE

You will be called again with this
Result: flight booked from Toronto to Ottawa at 2pm for 100 CAD

You then output:

Answer: Flight booked to Ottawa at 2pm today
""".strip()


In [60]:
import random

In [82]:
def weather(location) -> str:
  return f"{random.randint(-5,10)}"

In [62]:
def book(source, destination, time)-> str:
  return f"flight booked from {source} to {destination} at {time} for {random.randint(100, 500)} CAD"

In [98]:
agent = Agent(prompt)

In [99]:
response = agent("Book a flight going from Calgary to Edmonton by 5pm today")
print(response)

Thought: The user intends to travel from Calgary which is the source location, to Edmonton which is the destination location, by 5pm today. I need to find out what the temperature would be at 5pm. 
Action: weather: Edmonton, 5pm
PAUSE


In [107]:
response = weather("Edmonton")
print(response)

9


In [108]:
next_prompt = f"Result: The temperature is {response} degrees"


In [109]:
agent(next_prompt)

'Action: book: Calgary, Edmonton, 5pm\nPAUSE'

In [110]:
next_prompt = f"Result: {book("Calgary", "Edmonton", "5pm")} today"

In [111]:
agent(next_prompt)

'Answer: Flight booked to Edmonton at 5pm today.'